In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from imblearn.over_sampling import SMOTE
import chardet
import re
import emoji

In [2]:
# -------------------------------------------
# Load & preprocess hate speech dataset
# -------------------------------------------
Column_names = ["Code mixed text", "Hate or Non Hate speech"]
hatespeech_df = pd.read_csv("dataset/hate_speech.tsv", sep='\t', names=Column_names, header=None)
hatespeech_df = hatespeech_df[~hatespeech_df["Hate or Non Hate speech"].isin(['n', 'on'])]
hatespeech_df["HateOrNot"] = hatespeech_df["Hate or Non Hate speech"].map({"yes": 1, "no": 0})
hatespeech_df.dropna(inplace=True)
hatespeech_df.reset_index(drop=True, inplace=True)

# Clean the text
def clean_text(text):
    text = str(text).lower()
    text = emoji.replace_emoji(text, replace='')
    text = re.sub(r"http\S+|www\S+", '', text)
    text = re.sub(r"@\w+|#\w+", '', text)
    text = re.sub(r"[^\w\s]", '', text)
    return text

hatespeech_df["Cleaned Text"] = hatespeech_df["Code mixed text"].apply(clean_text)


In [3]:
# -------------------------------------------
# Load profanity list
# -------------------------------------------
with open("dataset/Hinglish_Profanity_List.csv", 'rb') as f:
    result = chardet.detect(f.read(10000))
encoding_type = result['encoding']

profanity_df = pd.read_csv("dataset/Hinglish_Profanity_List.csv",
                           encoding=encoding_type,
                           names=["Code mixed words", "English Meaning", "Severity scoring"],
                           header=None)
profanity_df["Code mixed words"] = profanity_df["Code mixed words"].astype(str).str.strip().str.lower()
profanity_dict = dict(zip(profanity_df["Code mixed words"], profanity_df["Severity scoring"]))

# Extract features
def extract_profanity_features(text, profanity_dict, max_len=100):
    words = text.split()
    count = 0
    severity_score = 0
    for word in words:
        if word in profanity_dict:
            count += 1
            severity_score += profanity_dict[word]
    binary_match = 1 if count > 0 else 0
    normalized_len = len(words) / max_len
    return count, severity_score, binary_match, normalized_len

hatespeech_df[["profanity_count", "profanity_score", "binary_profanity_match", "normalized_text_len"]] = \
    hatespeech_df["Cleaned Text"].apply(lambda x: pd.Series(extract_profanity_features(x, profanity_dict)))


In [4]:
# -------------------------------------------
# Tokenizer and Dataset
# -------------------------------------------
tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")

class HateSpeechDataset(Dataset):
    def __init__(self, texts, features, labels, tokenizer, max_len=128):
        self.texts = texts
        self.features = features
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(self.texts[idx],
                                  padding='max_length',
                                  truncation=True,
                                  max_length=self.max_len,
                                  return_tensors="pt")
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "features": torch.tensor(self.features[idx], dtype=torch.float32),
            "label": torch.tensor(self.labels[idx], dtype=torch.float32)
        }


In [5]:
# -------------------------------------------
# Model with BiGRU + Attention + Profanity
# -------------------------------------------
class AttentionPooling(nn.Module):
    def __init__(self, hidden_dim):
        super(AttentionPooling, self).__init__()
        self.attn = nn.Sequential(
            nn.Linear(hidden_dim, 128),
            nn.Tanh(),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        scores = self.attn(x)
        weights = torch.softmax(scores, dim=1)
        return (x * weights).sum(dim=1)

class HateClassifier(nn.Module):
    def __init__(self, transformer, rnn_hidden=256, feature_dim=4):
        super().__init__()
        self.transformer = transformer
        self.bigru = nn.GRU(input_size=768, hidden_size=rnn_hidden, batch_first=True, bidirectional=True)
        self.attn = AttentionPooling(rnn_hidden * 2)
        self.classifier = nn.Sequential(
            nn.Linear(rnn_hidden * 2 + feature_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 1)
        )

    def forward(self, input_ids, attention_mask, features):
        x = self.transformer(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        x, _ = self.bigru(x)
        x = self.attn(x)
        combined = torch.cat([x, features], dim=1)
        return self.classifier(combined).squeeze(1)


In [6]:
# -------------------------------------------
# Focal Loss
# -------------------------------------------
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=0.25):
        super(FocalLoss, self).__init__()
        self.gamma = gamma
        self.alpha = alpha

    def forward(self, logits, targets):
        BCE_loss = nn.BCEWithLogitsLoss(reduction='none')(logits, targets)
        pt = torch.exp(-BCE_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * BCE_loss
        return focal_loss.mean()


In [7]:
# -------------------------------------------
# Prepare data and training
# -------------------------------------------
texts = hatespeech_df["Cleaned Text"].tolist()
custom_features = hatespeech_df[["profanity_count", "profanity_score", "binary_profanity_match", "normalized_text_len"]].values
labels = hatespeech_df["HateOrNot"].values

# Apply SMOTE
X_combined = np.hstack([np.zeros((len(custom_features), 768)), custom_features])  # placeholder for SMOTE
X_res, y_res = SMOTE(random_state=42).fit_resample(X_combined, labels)
custom_features_res = X_res[:, -4:]
indices_res = np.random.choice(len(texts), len(custom_features_res), replace=True)
texts_res = [texts[i] for i in indices_res]

# Split
X_train, X_test, feat_train, feat_test, y_train, y_test = train_test_split(
    texts_res, custom_features_res, y_res, test_size=0.2, random_state=42)

train_dataset = HateSpeechDataset(X_train, feat_train, y_train, tokenizer)
test_dataset = HateSpeechDataset(X_test, feat_test, y_test, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16)


In [8]:
from tqdm import tqdm

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load transformer and freeze parameters to reduce memory usage
transformer = AutoModel.from_pretrained("xlm-roberta-base")
for param in transformer.parameters():
    param.requires_grad = False

# Instantiate model
model = HateClassifier(transformer).to(device)

# Loss and optimizer
criterion = FocalLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4)

# Adjust batch size to avoid memory issues
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=8)

# Train loop
EPOCHS = 3
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}")
    for batch in progress_bar:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        features = batch['features'].to(device)
        labels = batch['label'].to(device)

        outputs = model(input_ids, attention_mask, features)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        progress_bar.set_postfix(loss=loss.item())

    print(f"Epoch {epoch+1}: Avg Loss = {total_loss/len(train_loader):.4f}")


Epoch 1: 100%|██████████| 583/583 [07:06<00:00,  1.37it/s, loss=0.0435]


Epoch 1: Avg Loss = 0.0439


Epoch 2: 100%|██████████| 583/583 [07:04<00:00,  1.37it/s, loss=0.042] 


Epoch 2: Avg Loss = 0.0434


Epoch 3: 100%|██████████| 583/583 [07:10<00:00,  1.35it/s, loss=0.0443]

Epoch 3: Avg Loss = 0.0434


In [ ]:
# -------------------------------------------
# Evaluation
# -------------------------------------------
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        features = batch['features'].to(device)
        labels = batch['label'].cpu().numpy()
        logits = model(input_ids, attention_mask, features).cpu()
        preds = torch.sigmoid(logits) > 0.5
        all_preds.extend(preds.numpy())
        all_labels.extend(labels)

print(classification_report(all_labels, all_preds, target_names=["Non-Hate", "Hate"]))
